---

## Capstone · Part 2 — Trend, relate, and present the closures

One last return to **rural hospital closures**. In Part 1 you counted and compared them; now you bring the Session 2 toolkit — trends over time, a relationship, small multiples, and presentation polish — and finish with a figure ready for a slide.

**Driving question.** Did rural closures accelerate over time, do the hardest-hit states carry the heaviest disease burden, and how does access hold up where hospitals remain?

We start, as before, from the cleaned closures data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"

In [ ]:
closures = pd.read_csv(f"{BASE_URL}/rural_hospital_closures.csv")
closures = closures.drop(columns="Unnamed: 0")
closures["closure_year"] = closures["closure_year"].replace(1019, 2019)
closures["closure_type"] = (closures["closure_type"].str.strip().str.capitalize()
                            .replace({"Complet": "Complete", "Convertd": "Converted"}))
closures["beds"] = closures["beds"].fillna(closures["beds"].median())
q1, q3 = closures["beds"].quantile([0.25, 0.75]); iqr = q3 - q1
closures = closures[closures["beds"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
closures = closures.dropna(subset=["state"])
closures.shape

### Task 1 — The trend: did closures accelerate?

Count closures per `closure_year` and plot them as a line, fully labeled. In a comment, say whether closures sped up over the period.

In [ ]:
# Your work here
per_year = closures["closure_year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(per_year.index, per_year.values, marker="o")
ax.set_title("Rural hospital closures per year")
ax.set_xlabel("Year")
ax.set_ylabel("Closures")
fig.tight_layout()
plt.show()

<details>
<summary><b>Solution</b></summary>

```python
per_year = closures["closure_year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(per_year.index, per_year.values, marker="o")
ax.set_title("Rural hospital closures per year")
ax.set_xlabel("Year")
ax.set_ylabel("Closures")
fig.tight_layout()
plt.show()
# Yes: closures drift upward from a couple a year in the mid-2000s to high single
# digits in the 2010s-2020s, with a sharp spike around 2019.
```

**Why this works.** `value_counts().sort_index()` turns the year column into an ordered count series — a time trend — and a line reads acceleration far better than bars would. The line is the time-series skill from Notebook 5, on yearly stakes.

</details>

### Task 2 — The relationship: are the hardest-hit states the sickest?

To relate closures to health, we enrich the closures with county context. The cell below builds a state-level table — closures per state, and each state's population-weighted diabetes prevalence from the ACS/PLACES county data (`merge` and `groupby`, both from the pandas course), so you can focus on the chart.

In [ ]:
acs = pd.read_csv(f"{BASE_URL}/acs2017.csv")
places = pd.read_csv(f"{BASE_URL}/places.csv")
counties = acs.merge(places, left_on="CountyId", right_on="CountyFIPS", how="inner")

state_health = (counties.groupby("State")
                .apply(lambda g: np.average(g["hdiabetes"], weights=g["TotalPop"]),
                       include_groups=False)
                .rename("diabetes"))
per_state = closures["state"].value_counts().rename("closures")
state = pd.concat([per_state, state_health], axis=1, join="inner").reset_index(names="state")
state.head()

Now plot it. Draw a scatter with a linear trend line of **`diabetes` (x)** against **`closures` (y)**, fully labeled. In a comment, say whether states with a heavier diabetes burden tend to have lost more hospitals.

In [ ]:
# Your work here
fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=state, x="diabetes", y="closures",
            line_kws={"color": "crimson"}, ax=ax)
ax.set_title("State diabetes burden vs. rural closures")
ax.set_xlabel("Diabetes prevalence (%, population-weighted)")
ax.set_ylabel("Rural hospital closures")
fig.tight_layout()
plt.show()

<details>
<summary><b>Solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=state, x="diabetes", y="closures",
            line_kws={"color": "crimson"}, ax=ax)
ax.set_title("State diabetes burden vs. rural closures")
ax.set_xlabel("Diabetes prevalence (%, population-weighted)")
ax.set_ylabel("Rural hospital closures")
fig.tight_layout()
plt.show()
# Positive (about 0.44): states with higher diabetes prevalence tend to have lost more
# rural hospitals -- the places losing access are also the places that most need care.
```

**Why this works.** The enrichment turns two unrelated tables into one state-level frame, and `regplot` — the relationship tool from Notebook 4 — draws the association. The upward slope is the capstone's core finding: closures are not random; they concentrate where the chronic-disease burden is highest.

</details>

### Task 3 — Small multiples of remaining access

Where hospitals remain, how busy are the clinics? Load the daily `clinic_visits` and, using a **figure-level** function, draw the **weekly mean visits** as small multiples — one panel per `site`.

In [ ]:
visits = pd.read_csv(f"{BASE_URL}/clinic_visits.csv", parse_dates=["date"])
weekly = (visits.pivot_table(index="date", columns="site", values="visits")
          .resample("W").mean()
          .reset_index()
          .melt("date", var_name="site", value_name="visits"))
weekly.head()

In [ ]:
# Your work here
g = sns.relplot(data=weekly, x="date", y="visits", col="site",
                kind="line", height=3.5, aspect=1.1)
g.set_axis_labels("Week", "Mean visits per day")
g.set_titles("{col_name}")
g.figure.suptitle("Weekly clinic visits by site", y=1.03)
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=45)
plt.show()

<details>
<summary><b>Solution</b></summary>

```python
g = sns.relplot(data=weekly, x="date", y="visits", col="site",
                kind="line", height=3.5, aspect=1.1)
g.set_axis_labels("Week", "Mean visits per day")
g.set_titles("{col_name}")
g.figure.suptitle("Weekly clinic visits by site", y=1.03)
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=45)
plt.show()
```

**Why this works.** `relplot` is the figure-level counterpart to `lineplot`, so `col="site"` lays out one time-series panel per clinic on shared axes — the faceting skill from this notebook, applied to the capstone. Side-by-side panels make the three sites honestly comparable, and all three share the same mid-year dip.

</details>

### Task 4 — The finale: one presentation-ready figure

Bring it home. For the **Hilltop** site, plot daily visits with a **7-day rolling mean** on top, **annotate the five-day March outage** with a shaded span and label, set a presentation theme, and save the figure as a PNG at 150 dpi. This is the chart you would put on a slide.

In [ ]:
hill = visits[visits["site"] == "Hilltop"].set_index("date")["visits"]
# filters the visits DataFrame down to rows where site equals "Hilltop", 
# then makes date the index and pulls out the visits column, 
# leaving you with a Series of visit counts indexed by date.
hill

In [ ]:
# Your work here
sns.set_theme(style="whitegrid", context="talk")


fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hill.index, hill.values, linewidth=0.7, alpha=0.4, label="daily")
ax.plot(hill.index, hill.rolling(7).mean(), color="crimson", label="7-day mean")

start, end = pd.Timestamp("2025-03-10"), pd.Timestamp("2025-03-14")
ax.axvspan(start, end, color="crimson", alpha=0.15)
ax.annotate("5-day outage", xy=(start, ax.get_ylim()[1]),
            xytext=(8, -16), textcoords="offset points", color="crimson")

ax.set_title("Hilltop clinic: daily visits, 2025")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.legend()
fig.tight_layout()
fig.savefig("hilltop_2025.png", dpi=150, bbox_inches="tight")
plt.show()

sns.set_theme()   # reset so later work starts clean

<details>
<summary><b>Solution</b></summary>

```python
sns.set_theme(style="whitegrid", context="talk")

hill = visits[visits["site"] == "Hilltop"].set_index("date")["visits"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hill.index, hill.values, linewidth=0.7, alpha=0.4, label="daily")
ax.plot(hill.index, hill.rolling(7).mean(), color="crimson", label="7-day mean")

start, end = pd.Timestamp("2025-03-10"), pd.Timestamp("2025-03-14")
ax.axvspan(start, end, color="crimson", alpha=0.15)
ax.annotate("5-day outage", xy=(start, ax.get_ylim()[1]),
            xytext=(8, -16), textcoords="offset points", color="crimson")

ax.set_title("Hilltop clinic: daily visits, 2025")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.legend()
fig.tight_layout()
fig.savefig("hilltop_2025.png", dpi=150, bbox_inches="tight")
plt.show()

sns.set_theme()   # reset so later work starts clean
```

**Why this works.** Every piece is a skill from Session 2: the rolling mean (Notebook 5), the annotated span (Notebook 5), and the theme, labels, and save (this notebook). Together they turn a raw daily series into a figure that explains itself — the outage marked, the trend legible, the text sized for a room.

</details>

### The insight — and one surprise

**Insight.** Rural closures are not a finished story from the past: they *accelerated*, climbing from a couple a year to high single digits with a spike around 2019. And they are not random — they concentrate in the states carrying the heaviest diabetes burden, so the communities losing hospitals are the ones that most need chronic-disease care.

**One surprise.** It is diabetes prevalence, more than income, that tracks closures: the state-level correlation with the diabetes burden (about 0.44) runs stronger than the one with poverty (about 0.30). The clearest signal for *where* rural access is disappearing is not simply how poor a state is, but how sick it is.

**That closes the course.** You started with the Figure/Axes model, worked through distributions, categories, relationships, and time series, learned to facet and polish — and carried one real project, rural hospital access, from a clean table all the way to a figure ready for an audience.